In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path="/content/drive/MyDrive/BookVerse/datasets/books.csv"
books=pd.read_csv(path)

In [ ]:
print("Rows :",books.shape[0])
print("Columns :",books.shape[1])

Rows : 52478
Columns : 25


In [ ]:
books = books[
    [
        "bookId",
        "title",
        "author",
        "series",
        "genres",
        "description",
        "publisher",
        "language",
        "rating",
        "numRatings",
        "likedPercent",
        "coverImg"
    ]
]

In [ ]:
books.head()

,bookId,title,author,series,genres,description,publisher,language,rating,numRatings,likedPercent,coverImg
0,2767052-the-hunger-games,The Hunger Games,Suzanne Collins,The Hunger Games #1,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...",WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,Scholastic Press,English,4.33,6376780,96.0,https://i.gr-assets.com/images/S/compressed.ph...
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,"J.K. Rowling, Mary GrandPré (Illustrator)",Harry Potter #5,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...",There is a door at the end of a silent corrido...,Scholastic Inc.,English,4.50,2507623,98.0,https://i.gr-assets.com/images/S/compressed.ph...
2,2657.To_Kill_a_Mockingbird,To Kill a Mockingbird,Harper Lee,To Kill a Mockingbird,"['Classics', 'Fiction', 'Historical Fiction', ...",The unforgettable novel of a childhood in a sl...,Harper Perennial Modern Classics,English,4.28,4501075,95.0,https://i.gr-assets.com/images/S/compressed.ph...
3,1885.Pride_and_Prejudice,Pride and Prejudice,"Jane Austen, Anna Quindlen (Introduction)",NaN,"['Classics', 'Fiction', 'Romance', 'Historical...",Alternate cover edition of ISBN 9780679783268S...,Modern Library,English,4.26,2998241,94.0,https://i.gr-assets.com/images/S/compressed.ph...
4,41865.Twilight,Twilight,Stephenie Meyer,The Twilight Saga #1,"['Young Adult', 'Fantasy', 'Romance', 'Vampire...",About three things I was absolutely positive.\...,"Little, Brown and Company",English,3.60,4964519,78.0,https://i.gr-assets.com/images/S/compressed.ph...


In [ ]:
books.isnull().sum()

,0
bookId,0
title,0
author,0
series,29008
genres,0
description,1338
publisher,3696
language,3806
rating,0
numRatings,0


In [ ]:
text_columns = ["title","author","series","genres","description","publisher"]

for col in text_columns:
    books[col] = (books[col].fillna("").astype(str).str.strip())
books["language"] = (books["language"].fillna("").astype(str).str.strip())

In [ ]:
print("Duplicate Title + Author:",books.duplicated(subset=["title","author"]).sum())
books = books.drop_duplicates(subset=["title","author"],keep="first")
print("Dataset Shape:", books.shape)

Duplicate Title + Author: 0
Dataset Shape: (42309, 13)


In [ ]:
books.isnull().sum()

,0
bookId,0
title,0
author,0
series,0
genres,0
description,0
publisher,0
language,0
rating,0
numRatings,0


In [ ]:
books = books[
    books["language"]
    .str.lower()
    .eq("english")
].copy()

print("English books:", len(books))

English books: 42607


In [ ]:
books["language"].value_counts().head(20)

,count
language,
English,42607


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\[.*?\]", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
print(books.shape)

(42607, 12)


In [ ]:
text_columns = ["title","author","series","description","publisher"]
for col in text_columns:
    books[col] = books[col].apply(clean_text)

In [ ]:
books["genres"] = (
    books["genres"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

In [ ]:
books.head()

,bookId,title,author,series,genres,description,publisher,language,rating,numRatings,likedPercent,coverImg,combined_features
0,2767052-the-hunger-games,the hunger games,suzanne collins,the hunger games,,winning means fame and fortune losing means ce...,scholastic press,English,4.33,6376780,96.0,https://i.gr-assets.com/images/S/compressed.ph...,winning means fame and fortune losing mea...
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,harry potter and the order of the phoenix,j k rowling mary grandpr illustrator,harry potter,,there is a door at the end of a silent corrido...,scholastic inc,English,4.50,2507623,98.0,https://i.gr-assets.com/images/S/compressed.ph...,there is a door at the end of a silent co...
2,2657.To_Kill_a_Mockingbird,to kill a mockingbird,harper lee,to kill a mockingbird,,the unforgettable novel of a childhood in a sl...,harper perennial modern classics,English,4.28,4501075,95.0,https://i.gr-assets.com/images/S/compressed.ph...,the unforgettable novel of a childhood in...
3,1885.Pride_and_Prejudice,pride and prejudice,jane austen anna quindlen introduction,,,alternate cover edition of isbn since its imme...,modern library,English,4.26,2998241,94.0,https://i.gr-assets.com/images/S/compressed.ph...,alternate cover edition of isbn since its...
4,41865.Twilight,twilight,stephenie meyer,the twilight saga,,about three things i was absolutely positive f...,little brown and company,English,3.60,4964519,78.0,https://i.gr-assets.com/images/S/compressed.ph...,about three things i was absolutely posit...


In [ ]:
#weighted feature engineering.
books["combined_features"] = (
    (books["genres"] + " ") * 5 +
    (books["description"] + " ") * 4 +
    (books["series"] + " ") * 2 +
    (books["title"] + " ") * 3 +
    (books["author"] + " ") * 2 +
    books["publisher"]
)

In [ ]:
books["combined_features"].head()

,combined_features
0,"['young adult', 'fiction', 'dystopia', 'fantas..."
1,"['fantasy', 'young adult', 'fiction', 'magic',..."
2,"['classics', 'fiction', 'historical fiction', ..."
3,"['classics', 'fiction', 'romance', 'historical..."
4,"['young adult', 'fantasy', 'romance', 'vampire..."


In [ ]:
books[["title", "genres", "combined_features"]].head(5)

,title,genres,combined_features
0,the hunger games,"['young adult', 'fiction', 'dystopia', 'fantas...","['young adult', 'fiction', 'dystopia', 'fantas..."
1,harry potter and the order of the phoenix,"['fantasy', 'young adult', 'fiction', 'magic',...","['fantasy', 'young adult', 'fiction', 'magic',..."
2,to kill a mockingbird,"['classics', 'fiction', 'historical fiction', ...","['classics', 'fiction', 'historical fiction', ..."
3,pride and prejudice,"['classics', 'fiction', 'romance', 'historical...","['classics', 'fiction', 'romance', 'historical..."
4,twilight,"['young adult', 'fantasy', 'romance', 'vampire...","['young adult', 'fantasy', 'romance', 'vampire..."


In [ ]:
books.isnull().sum()

,0
bookId,0
title,0
author,0
series,0
genres,0
description,0
publisher,0
language,0
rating,0
numRatings,0


In [ ]:
print("Final Dataset Shape :", books.shape)
print("Missing Combined Features:",
      books["combined_features"].isna().sum())
print("Duplicate Books:",
      books.duplicated(
          subset=["title","author"]
      ).sum())

Final Dataset Shape : (42309, 13)
Missing Combined Features: 0
Duplicate Books: 0


In [ ]:
books = (
    books
    .drop_duplicates(subset=["title", "author"], keep="first")
    .reset_index(drop=True)
)
print("Final duplicate count:",
      books.duplicated(["title", "author"]).sum())

Final duplicate count: 0


In [ ]:
books.to_csv("/content/drive/MyDrive/BookVerse/datasets/books_cleaned.csv",index=False)